# Apply Z-Severance Cuts on MDIO

```{article-info}
:author: Brian Michell
:date: "{sub-ref}`today`"
:read-time: "{sub-ref}`wordcount-minutes` min read"
:class-container: sd-p-0 sd-outline-muted sd-rounded-3 sd-font-weight-light
```

A z-severance cut restricts specific vertical time or depth intervals within one or
more spatial regions of a seismic survey (for example, removing 2–5 seconds inside a
mineral-rights boundary while keeping 0–2 seconds and 5 seconds to bottom). In MDIO,
a z-severance mask is an N-D boolean variable matching the default seismic variable's
dimensions, lazily initialized to `False` (`fill_value=False`). `True` entries mark
samples that must be zeroed out on export.

Each severance mask carries canonical JSON attributes on the variable:

- `maskType`: `"z_severance"`
- `createdOn`: UTC ISO-8601 timestamp used to select the latest mask by default
- `metadataLock`: `true` while external business logic is populating the mask, and
  `false` once finalized

```{note}
Exports fail closed: if a severance mask is left with `metadataLock: true` (for
example, if a polygon job crashes mid-write), `resolve_z_severance_mask` and
`mdio_to_segy` raise `MDIOSeveranceError` rather than exporting an incomplete cut.
```


## Create a sample 3D MDIO dataset

Build a small 3D post-stack dataset with `inline`, `crossline`, and `time` (0 to
7000 ms in 1000 ms steps), populated `cdp_x` / `cdp_y` coordinates, and constant
non-zero amplitudes.


In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import zarr

from mdio import apply_z_severance_mask
from mdio import open_mdio
from mdio import resolve_z_severance_mask
from mdio import to_mdio
from mdio import z_severance_mask_session

mdio_path = Path("severance_demo.mdio")

inlines = np.array([100, 101, 102, 103], dtype=np.int32)
crosslines = np.array([200, 201, 202, 203], dtype=np.int32)
times = np.arange(0, 8000, 1000, dtype=np.int32)  # 0..7000 ms

cdp_x, cdp_y = np.meshgrid(
    np.linspace(500_000.0, 500_300.0, inlines.size),
    np.linspace(4_000_000.0, 4_000_300.0, crosslines.size),
    indexing="ij",
)

dataset = xr.Dataset(
    {
        "amplitude": (("inline", "crossline", "time"), np.ones((4, 4, 8), dtype=np.float32)),
        "trace_mask": (("inline", "crossline"), np.ones((4, 4), dtype=bool)),
    },
    coords={
        "inline": ("inline", inlines),
        "crossline": ("crossline", crosslines),
        "time": ("time", times),
        "cdp_x": (("inline", "crossline"), cdp_x),
        "cdp_y": (("inline", "crossline"), cdp_y),
    },
    attrs={"attributes": {"defaultVariableName": "amplitude"}, "name": "severance-demo"},
)
to_mdio(dataset, mdio_path, mode="w")

## Define custom severance business logic

Your business logic operates on the dataset's CDP coordinates (`cdp_x`, `cdp_y`) and
vertical coordinate (`time` or `depth`), writing `True` into the Zarr mask array
wherever samples must be severed.

Using bitwise OR (`|=`) when writing to the mask array lets you combine multiple
polygons and discrete or continuous vertical windows into the same mask without
clobbering earlier cuts.


In [ ]:
def sever_region_by_cdp(
    ds: xr.Dataset,
    mask_array: zarr.Array,
    *,
    x_bounds: tuple[float, float],
    y_bounds: tuple[float, float],
    time_intervals: list[tuple[float, float]],
) -> None:
    """Mark (cdp_x, cdp_y, time) samples inside the bounding box and time intervals as True."""
    cdp_x_vals = ds["cdp_x"].values
    cdp_y_vals = ds["cdp_y"].values
    time_vals = ds["time"].values

    spatial_hit = (
        (cdp_x_vals >= x_bounds[0])
        & (cdp_x_vals <= x_bounds[1])
        & (cdp_y_vals >= y_bounds[0])
        & (cdp_y_vals <= y_bounds[1])
    )

    vertical_hit = np.zeros_like(time_vals, dtype=bool)
    for t_start, t_end in time_intervals:
        vertical_hit |= (time_vals >= t_start) & (time_vals <= t_end)

    # Broadcast spatial selection across any intermediate gather dims and vertical axis
    extra_dims = mask_array.ndim - spatial_hit.ndim - 1
    expanded_spatial = spatial_hit.reshape(*spatial_hit.shape, *([1] * (extra_dims + 1)))
    cut_volume = expanded_spatial & vertical_hit

    mask_array[:] = np.asarray(mask_array[:]) | cut_volume

## Create the initial severance mask (`v1`)

`z_severance_mask_session` creates the mask variable in a locked state
(`metadataLock: true`), yields the writable Zarr array to your business logic, and
releases the lock (`metadataLock: false`) only when the `with` block exits cleanly.

Here we sever 2000–5000 ms over the southwest quadrant (`cdp_x <= 500_100`,
`cdp_y <= 4_000_100`).


In [ ]:
ds = open_mdio(mdio_path)

with z_severance_mask_session(mdio_path, "z_severance_v1") as mask_v1:
    sever_region_by_cdp(
        ds,
        mask_v1,
        x_bounds=(500_000.0, 500_100.0),
        y_bounds=(4_000_000.0, 4_000_100.0),
        time_intervals=[(2000.0, 5000.0)],
    )

ds = open_mdio(mdio_path)
print("attrs:", ds["z_severance_v1"].attrs)
print("trace (0, 0) mask:", ds["z_severance_v1"].values[0, 0])

## Progressively update severance masks

As new mineral-rights restrictions or polygon amendments arrive, you can progressively
build on existing cuts in two ways:

1. **Versioned masks (recommended):** Create a new mask variable (for example,
   `z_severance_v2`), copy the prior mask's values as your starting baseline, and
   apply additional polygons or depth intervals. Because `z_severance_v2` has a newer
   `createdOn` timestamp, it automatically becomes the default mask for export while
   preserving `z_severance_v1` for auditability.
2. **In-place overwrite:** Pass `overwrite=True` to `z_severance_mask_session` to
   replace a mask in place under the same name.

Below, we seed `z_severance_v2` from `z_severance_v1` and add two more cuts: a
shallow band (0–1000 ms) and a deep band (6000–7000 ms) in the northeast quadrant.


In [ ]:
ds = open_mdio(mdio_path)

with z_severance_mask_session(mdio_path, "z_severance_v2") as mask_v2:
    # Start from the existing v1 severance mask
    mask_v2[:] = ds["z_severance_v1"].values

    # Progressively add new polygons / discrete time windows
    sever_region_by_cdp(
        ds,
        mask_v2,
        x_bounds=(500_200.0, 500_300.0),
        y_bounds=(4_000_200.0, 4_000_300.0),
        time_intervals=[(0.0, 1000.0), (6000.0, 7000.0)],
    )

ds = open_mdio(mdio_path)
print("default resolved mask:", resolve_z_severance_mask(ds))
print("SW trace (0, 0) in v2:", ds["z_severance_v2"].values[0, 0])
print("NE trace (3, 3) in v2:", ds["z_severance_v2"].values[3, 3])

## Apply masks in Python or during SEG-Y export

When `mdio_to_segy` runs, it calls `resolve_z_severance_mask` and zeroes out all
`True` samples before writing traces:

```python
# Automatically applies the latest mask by createdOn ("z_severance_v2")
mdio_to_segy(segy_spec, mdio_path, "output_latest.sgy")

# Or export against a specific mask version
mdio_to_segy(segy_spec, mdio_path, "output_v1.sgy", severance_mask="z_severance_v1")
```

Backend consumers working directly with Xarray datasets use the same pair of
functions:


In [ ]:
# Latest mask (z_severance_v2)
latest_mask = resolve_z_severance_mask(ds)
ds_latest = apply_z_severance_mask(ds, latest_mask)
print("v2 SW trace (0, 0) amplitudes:", ds_latest["amplitude"].values[0, 0])
print("v2 NE trace (3, 3) amplitudes:", ds_latest["amplitude"].values[3, 3])

# Explicit v1 mask (NE trace is still unsevered in v1)
v1_mask = resolve_z_severance_mask(ds, severance_mask="z_severance_v1")
ds_v1 = apply_z_severance_mask(ds, v1_mask)
print("v1 NE trace (3, 3) amplitudes:", ds_v1["amplitude"].values[3, 3])

## Recap

- Use `z_severance_mask_session` to create an N-D boolean mask initialized to `False`
  with `metadataLock: true` and `createdOn` metadata.
- Populate severed sections with `True` using your CDP/time/depth business logic;
  the lock is released automatically when the `with` block completes without error.
- Progressively update cuts by seeding a new versioned mask from the prior mask (or
  updating with `overwrite=True`) and OR-combining (`|=`) new polygon/depth slices.
- `mdio_to_segy` (and `resolve_z_severance_mask`) selects the latest mask by
  `createdOn` by default, or accepts `severance_mask="<name>"` to target a specific
  version.
